# Day 02c: Embeddings — From Integers to Vectors

**Time:** ~20 minutes  
**Prerequisites:** Day 02a (model weights), Day 02b (tokenizer)  
**Goal:** Understand how token IDs become the 768-dimensional vectors the model works with.

---

So far: we can parse model weights (02a) and tokenize text into IDs (02b).  
But the model doesn't do math on integers. It needs **vectors** — dense arrays of floating-point numbers.

Embeddings are the bridge: `integer ID -> 768-dim vector`.

Think of it like DNS: a domain name (`cat`) is human-readable but useless to a router.  
DNS resolves it to an IP address (`192.168.1.42`) that the network can route.  
Embeddings resolve a token ID (`9246`) to a vector that the neural network can route through its layers.

## Setup: Load Weights + Tokenizer from Previous Notebooks

We need the model weights and the tokenizer. Same code as 02a and 02b — copy-pasted for self-containment.

In [ ]:
import json
import struct
import os
import re
import numpy as np

MODEL_DIR = "gpt2_weights"

# --- Load weights (from 02a) ---
def parse_safetensors(path):
    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(header_len))
        data_start = 8 + header_len
        tensors = {}
        for name, info in header.items():
            if name == "__metadata__":
                continue
            dtype = {"F32": np.float32, "F16": np.float16}[info["dtype"]]
            start, end = info["data_offsets"]
            f.seek(data_start + start)
            tensors[name] = np.frombuffer(f.read(end - start), dtype=dtype).reshape(info["shape"])
        return tensors

weights = parse_safetensors(os.path.join(MODEL_DIR, "model.safetensors"))

# --- Load tokenizer (from 02b) ---
with open(os.path.join(MODEL_DIR, "vocab.json")) as f:
    vocab = json.load(f)
with open(os.path.join(MODEL_DIR, "merges.txt")) as f:
    merges_raw = f.read().strip().split("\n")[1:]

id_to_token = {v: k for k, v in vocab.items()}
bpe_ranks = {tuple(m.split()): i for i, m in enumerate(merges_raw)}

def bytes_to_unicode():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}
GPT2_PATTERN = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+""")

def get_pairs(tokens):
    return set(zip(tokens[:-1], tokens[1:]))

def bpe_merge(tokens):
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best not in bpe_ranks:
            break
        first, second = best
        new = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == first and tokens[i+1] == second:
                new.append(first + second)
                i += 2
            else:
                new.append(tokens[i])
                i += 1
        tokens = new
    return tokens

def encode(text):
    ids = []
    for chunk in re.findall(GPT2_PATTERN, text):
        uc = [byte_encoder[b] for b in chunk.encode("utf-8")]
        for t in bpe_merge(uc):
            ids.append(vocab[t])
    return ids

def decode(token_ids):
    text = "".join(id_to_token[i] for i in token_ids)
    return bytes([byte_decoder[c] for c in text]).decode("utf-8", errors="replace")

print(f"Weights loaded: {len(weights)} tensors")
print(f"Tokenizer ready: {len(vocab):,} tokens")

## Step 1: Token Embeddings — The Lookup Table

The token embedding matrix `wte.weight` has shape `[50257, 768]`.

- **50,257 rows** — one for every token in the vocabulary
- **768 columns** — the embedding dimension (d_model)

To embed a token, you just **index into this matrix**. Token ID 9246 → row 9246.  
No multiplication needed — it's a pure table lookup, like indexing into an array.

```
wte.weight:   50257 rows x 768 columns

           col0    col1    col2    ...   col767
row 0    [ 0.012, -0.023,  0.045, ...,  0.008 ]   <- token 0
row 1    [-0.034,  0.011, -0.007, ...,  0.015 ]   <- token 1
...
row 9246 [ 0.041, -0.019,  0.033, ..., -0.028 ]   <- "cat"
...
row 50256[ 0.003,  0.001, -0.002, ...,  0.001 ]   <- <|endoftext|>
```

Think of it like a hash map: `embedding = wte[token_id]`

In [ ]:
wte = weights["wte.weight"]  # Token embedding table

print(f"Token embedding matrix (wte.weight)")
print(f"  Shape: {wte.shape}")
print(f"  Dtype: {wte.dtype}")
print(f"  Size:  {wte.nbytes / (1024*1024):.1f} MB")
print()

# Look up embeddings for a sentence
text = "The cat sat"
token_ids = encode(text)

print(f"Text: '{text}'")
print(f"Token IDs: {token_ids}")
print()

for tid in token_ids:
    embedding = wte[tid]  # Just array indexing!
    token_str = decode([tid])
    print(f"  Token '{token_str}' (ID {tid}):")
    print(f"    Vector: [{embedding[0]:.4f}, {embedding[1]:.4f}, {embedding[2]:.4f}, ..., {embedding[-1]:.4f}]")
    print(f"    Shape:  {embedding.shape}")
    print(f"    Norm:   {np.linalg.norm(embedding):.4f}")

## Step 2: Position Embeddings — Where Are You in the Sequence?

Token embeddings tell the model **what** a token is, but not **where** it appears.  
"The cat sat" and "sat cat The" would produce the same token embeddings in different order.

Position embeddings fix this. `wpe.weight` has shape `[1024, 768]`:
- **1024 rows** — one for each possible position (GPT-2's max sequence length)
- **768 columns** — same dimension as token embeddings (so they can be added)

The final input to the model is: `token_embedding + position_embedding`

```
 Position 0: wte["The"]  + wpe[0]  = input vector for "The" at position 0
 Position 1: wte["cat"]  + wpe[1]  = input vector for "cat" at position 1
 Position 2: wte["sat"]  + wpe[2]  = input vector for "sat" at position 2
```

Think of it like: token embedding = the package contents, position embedding = the sequence number.

In [ ]:
wpe = weights["wpe.weight"]  # Position embedding table

print(f"Position embedding matrix (wpe.weight)")
print(f"  Shape: {wpe.shape}")
print(f"  Max sequence length: {wpe.shape[0]}")
print()

# Build the full input embeddings for "The cat sat"
token_ids = encode("The cat sat")
seq_len = len(token_ids)

token_embeddings = wte[token_ids]        # (seq_len, 768) — what each token is
position_embeddings = wpe[:seq_len]       # (seq_len, 768) — where each token is
input_embeddings = token_embeddings + position_embeddings  # element-wise add

print(f"Token embeddings shape:    {token_embeddings.shape}")
print(f"Position embeddings shape: {position_embeddings.shape}")
print(f"Input embeddings shape:    {input_embeddings.shape}")
print()
print("This is the actual input to the first transformer block.")
print("A matrix of shape (seq_len, 768) — one 768-dim vector per token.")

## Step 3: Do Embeddings Capture Meaning?

These 768-dimensional vectors aren't random — they were learned during training.  
Similar words should have similar vectors. Let's check using **cosine similarity**:

$$\text{similarity}(a, b) = \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

Result is between -1 (opposite) and +1 (identical direction).  
Think of it like: two vectors pointing the same direction = similar meaning.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_embedding(word):
    """Get the token embedding for a single-token word."""
    ids = encode(word)
    if len(ids) == 1:
        return wte[ids[0]]
    # For multi-token words, average the embeddings
    return np.mean(wte[ids], axis=0)

# Compare word pairs
pairs = [
    (" king", " queen"),
    (" king", " throne"),
    (" king", " potato"),
    (" cat", " dog"),
    (" cat", " server"),
    (" fast", " quick"),
    (" fast", " slow"),
    (" GPU", " CPU"),
    (" GPU", " sandwich"),
]

print(f"{'Word A':<12s} {'Word B':<12s} {'Similarity':>10s}  Interpretation")
print("-" * 65)
for a, b in pairs:
    sim = cosine_similarity(get_embedding(a), get_embedding(b))
    bar = "#" * int(max(0, sim) * 30)
    label = "high" if sim > 0.5 else "medium" if sim > 0.2 else "low"
    print(f"{a:<12s} {b:<12s} {sim:>10.4f}  {bar} ({label})")

## Step 4: Position Embeddings — Do Nearby Positions Look Similar?

Intuition says positions close together should have more similar embeddings than positions far apart.  
Let's verify: how does similarity decay with distance?

In [ ]:
# Compare position 0 to positions 1, 10, 100, 500, 1000
pos0 = wpe[0]

print("Position embedding similarity (relative to position 0):")
print(f"{'Position':<12s} {'Distance':>10s} {'Cosine Sim':>12s}")
print("-" * 38)

for pos in [1, 2, 5, 10, 50, 100, 500, 1023]:
    sim = cosine_similarity(pos0, wpe[pos])
    bar = "#" * int(max(0, (sim + 1) / 2) * 20)  # normalize to 0-1 for display
    print(f"  pos {pos:<6d} {pos:>10d} {sim:>12.4f}  {bar}")

print()
print("Nearby positions are more similar — the model can sense distance.")
print("This is how the model knows word order without explicit position markers.")

## Step 5: Putting It Together — The Full Embedding Pipeline

Here's the complete embedding function that will be the first step of our forward pass:

```
"The cat sat on the mat"
         │
   encode()          (tokenizer)
         │
  [464, 9246, 3332, 319, 262, 2603]
         │
   wte[ids]          (token lookup — what each token is)
         │
   + wpe[:seq_len]   (position lookup — where each token is)
         │
   (6, 768) matrix   (ready for transformer blocks)
```

In [ ]:
def embed(token_ids, wte, wpe):
    """Convert token IDs to input embeddings.
    
    Like DNS resolution: token ID -> vector the model can route.
    
    token_ids: list of integers
    wte: token embedding matrix [vocab_size, d_model]
    wpe: position embedding matrix [max_seq_len, d_model]
    returns: numpy array of shape (seq_len, d_model)
    """
    seq_len = len(token_ids)
    token_emb = wte[token_ids]      # (seq_len, 768)
    pos_emb = wpe[:seq_len]         # (seq_len, 768)
    return token_emb + pos_emb      # (seq_len, 768)

# Test it end-to-end
text = "The cat sat on the mat"
token_ids = encode(text)
embeddings = embed(token_ids, wte, wpe)

print(f"Input text:     '{text}'")
print(f"Token IDs:      {token_ids}")
print(f"Embedding shape: {embeddings.shape}")
print()
print(f"Each row is a 768-dimensional vector.")
print(f"This {embeddings.shape[0]}x{embeddings.shape[1]} matrix is the input to transformer block h.0.")
print()
print(f"Memory for this input: {embeddings.nbytes} bytes ({embeddings.nbytes/1024:.1f} KB)")
print(f"Memory for the embedding tables: {(wte.nbytes + wpe.nbytes) / (1024*1024):.1f} MB")

## Key Takeaways

- **Token embedding is a table lookup**, not a matrix multiplication. `wte[token_id]` gives you the vector. Like an array index.
- **Position embedding adds location awareness**. Without it, the model can't tell word order.
- **Both tables are learned** during training — similar words end up with similar vectors.
- The input to the transformer is: `wte[token_ids] + wpe[:seq_len]` — a `(seq_len, 768)` matrix.
- **Embedding tables take memory**: GPT-2's wte is ~147 MB. Larger vocabs (128K+) in modern models take even more.

## Next

**Notebook 04** — We take this `(seq_len, 768)` matrix and pass it through **attention** — the core operation where tokens look at each other.

## References

- *Inference Engineering* Ch 2.2.2 (pp. 50–51) — Embedding layer, transformer block structure
- *Inference Engineering* Ch 2.2.1 (pp. 49–50) — Model architecture, config.json